# Sales Data Analysis Dashboard

**Project type:** Exploratory data analysis and interactive visualizations  
**Dataset:** `sales_dataset.csv`  
**Tools:** Python, Pandas, NumPy, Plotly, Jupyter / Google Colab

## Project objective
Explore e-commerce sales data, summarize key performance indicators (KPIs), compare revenue across product categories and regions, and inspect the distribution of revenue by payment method and category.

## Notebook workflow
1. Load the dataset from the project folder (or upload it in Google Colab).
2. Inspect data quality and prepare the date column.
3. Calculate sales KPIs.
4. Build four interactive Plotly visualizations.
5. Summarize evidence-based findings and business questions to investigate.

> **Data-quality note:** The supplied dates span **2022-01-01 to 2035-09-09**. Confirm that this range is intentional before presenting the date trend as real historical performance. The notebook reports the date range rather than assuming it is valid.


## 1. Import libraries

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Load the dataset

Place `sales_dataset.csv` in the same folder as this notebook. If the file is not found and the notebook is running in Google Colab, the cell will ask you to upload it.

In [ ]:
DATA_FILE = Path("sales_dataset.csv")

if DATA_FILE.exists():
    data = pd.read_csv(DATA_FILE)
else:
    # Convenience fallback for Google Colab users.
    try:
        from google.colab import files
        uploaded = files.upload()
        matching_files = [name for name in uploaded if Path(name).name == "sales_dataset.csv"]
        if not matching_files:
            raise FileNotFoundError(
                "Please upload a file named 'sales_dataset.csv' and rerun this cell."
            )
        data = pd.read_csv(io.BytesIO(uploaded[matching_files[0]]))
    except ImportError as exc:
        raise FileNotFoundError(
            "Could not find sales_dataset.csv. Put it beside this notebook, "
            "or upload it when running in Google Colab."
        ) from exc

required_columns = [
    "order_id", "order_date", "customer_id", "product_category", "region",
    "quantity", "unit_price", "discount", "payment_method", "delivery_days",
    "customer_rating", "revenue"
]
missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

# Parse dates safely; invalid dates remain NaT and are reported in the quality checks.
data["order_date"] = pd.to_datetime(data["order_date"], errors="coerce")

print(f"Dataset loaded successfully: {data.shape[0]:,} rows × {data.shape[1]} columns")
display(data.head())


## 3. Data inspection and quality checks

In [ ]:
print("### Dataset structure")
data.info()

print("\n### Summary statistics")
display(data.describe(include="all").T)

quality_summary = pd.DataFrame({
    "missing_values": data.isna().sum(),
    "missing_percent": (data.isna().mean() * 100).round(2),
    "data_type": data.dtypes.astype(str),
})
display(quality_summary)

print(f"Exact duplicate rows: {data.duplicated().sum():,}")
print(f"Duplicate order IDs: {data['order_id'].duplicated().sum():,}")
print(f"Invalid / missing order dates: {data['order_date'].isna().sum():,}")

valid_dates = data["order_date"].dropna()
if not valid_dates.empty:
    print(f"Order date range: {valid_dates.min():%Y-%m-%d} to {valid_dates.max():%Y-%m-%d}")
    if valid_dates.max() > pd.Timestamp.today().normalize():
        print("DATA CHECK: Some order dates are in the future. Verify the source before interpreting time trends.")

print("\n### Category and region values")
display(data[["product_category", "region", "payment_method"]].nunique().rename("unique_values").to_frame())


## 4. Key performance indicators (KPIs)

These KPIs summarize the values present in the dataset. They do not by themselves establish profitability, growth, or causation.

In [ ]:
total_revenue = data["revenue"].sum()
total_quantity = data["quantity"].sum()
avg_rating = data["customer_rating"].mean()
avg_discount = data["discount"].mean() * 100
average_order_revenue = data["revenue"].mean()
average_delivery_days = data["delivery_days"].mean()

kpis = pd.DataFrame({
    "Metric": [
        "Total revenue",
        "Total units sold",
        "Average customer rating",
        "Average discount",
        "Average revenue per record",
        "Average delivery time (days)",
    ],
    "Value": [
        f"${total_revenue:,.2f}",
        f"{total_quantity:,.0f}",
        f"{avg_rating:.2f} / 5",
        f"{avg_discount:.1f}%",
        f"${average_order_revenue:,.2f}",
        f"{average_delivery_days:.2f}",
    ],
})
display(kpis)


## 5. Interactive dashboard visualizations

The original four visualizations are retained:
- Revenue trend over time
- Revenue by product category
- Revenue distribution by region
- Revenue by payment method and product category


In [ ]:
def plot_revenue_trends(df: pd.DataFrame) -> go.Figure:
    """Show daily revenue over time, excluding rows with invalid dates."""
    trend_data = (
        df.dropna(subset=["order_date"])
          .groupby("order_date", as_index=False)["revenue"]
          .sum()
          .sort_values("order_date")
    )
    fig = px.line(
        trend_data,
        x="order_date",
        y="revenue",
        title="Daily Revenue Performance",
        labels={"order_date": "Order date", "revenue": "Revenue ($)"},
        template="plotly_white",
    )
    fig.update_traces(line_color="#1f77b4", line_width=1.5, hovertemplate="%{x|%Y-%m-%d}<br>Revenue: $%{y:,.2f}<extra></extra>")
    fig.update_layout(hovermode="x unified")
    return fig


def plot_category_performance(df: pd.DataFrame) -> go.Figure:
    """Compare total revenue by product category."""
    category_data = (
        df.groupby("product_category", as_index=False)["revenue"]
          .sum()
          .sort_values("revenue", ascending=True)
    )
    fig = px.bar(
        category_data,
        x="revenue",
        y="product_category",
        orientation="h",
        title="Revenue Contribution by Product Category",
        labels={"revenue": "Total revenue ($)", "product_category": "Category"},
        color="revenue",
        color_continuous_scale="Blues",
        template="plotly_white",
    )
    fig.update_layout(coloraxis_showscale=False, yaxis_title="Product category")
    fig.update_traces(hovertemplate="%{y}<br>Revenue: $%{x:,.2f}<extra></extra>")
    return fig


def plot_regional_distribution(df: pd.DataFrame) -> go.Figure:
    """Show each region's share of total revenue."""
    region_data = df.groupby("region", as_index=False)["revenue"].sum()
    fig = px.pie(
        region_data,
        values="revenue",
        names="region",
        title="Revenue Distribution by Region",
        hole=0.45,
        color_discrete_sequence=px.colors.qualitative.Prism,
        template="plotly_white",
    )
    fig.update_traces(textposition="inside", textinfo="percent+label", hovertemplate="%{label}<br>Revenue: $%{value:,.2f}<br>Share: %{percent}<extra></extra>")
    return fig


def plot_payment_breakdown(df: pd.DataFrame) -> go.Figure:
    """Show revenue by payment method and product category."""
    fig = px.sunburst(
        df,
        path=["payment_method", "product_category"],
        values="revenue",
        title="Revenue Breakdown: Payment Methods and Categories",
        color_discrete_sequence=px.colors.qualitative.Pastel,
    )
    fig.update_traces(hovertemplate="%{label}<br>Revenue: $%{value:,.2f}<extra></extra>")
    return fig


fig_trend = plot_revenue_trends(data)
fig_category = plot_category_performance(data)
fig_region = plot_regional_distribution(data)
fig_payment = plot_payment_breakdown(data)

fig_trend.show()
fig_category.show()
fig_region.show()
fig_payment.show()


## 6. Findings and business questions

In [ ]:
# Calculate findings from the loaded data so the notebook stays accurate if the CSV changes.
category_revenue = data.groupby("product_category")["revenue"].sum().sort_values(ascending=False)
region_revenue = data.groupby("region")["revenue"].sum().sort_values(ascending=False)
payment_revenue = data.groupby("payment_method")["revenue"].sum().sort_values(ascending=False)
category_rating = data.groupby("product_category")["customer_rating"].mean().sort_values()

if not category_revenue.empty:
    top_category = category_revenue.index[0]
    top_category_value = category_revenue.iloc[0]
    category_share = top_category_value / total_revenue * 100 if total_revenue else np.nan
    print(f"Highest-revenue category: {top_category} (${top_category_value:,.2f}; {category_share:.1f}% of recorded revenue)")

if not region_revenue.empty:
    top_region = region_revenue.index[0]
    top_region_value = region_revenue.iloc[0]
    region_share = top_region_value / total_revenue * 100 if total_revenue else np.nan
    print(f"Highest-revenue region: {top_region} (${top_region_value:,.2f}; {region_share:.1f}% of recorded revenue)")

if not payment_revenue.empty:
    top_payment = payment_revenue.index[0]
    top_payment_value = payment_revenue.iloc[0]
    payment_share = top_payment_value / total_revenue * 100 if total_revenue else np.nan
    print(f"Highest-revenue payment method: {top_payment} (${top_payment_value:,.2f}; {payment_share:.1f}% of recorded revenue)")

print(f"Average customer rating: {avg_rating:.2f}/5")
print(f"Average delivery time: {average_delivery_days:.2f} days")

print("\nAverage rating by product category:")
display(category_rating.rename("average_customer_rating").to_frame().round(2))

print("\nBusiness questions worth investigating:")
print("1. What factors contribute to the relatively low or high customer ratings?")
print("2. Are delivery times associated with customer ratings?")
print("3. Do discounts increase units sold or revenue enough to justify their cost?")
print("4. Are regional revenue differences driven by order volume, product mix, or average order value?")
print("5. After verifying the date range, are there reliable seasonal or long-term revenue patterns?")


## 7. Initial recommendations

Use the results above to decide which actions are justified. The following are **hypotheses to test**, not proven causes:

- **Customer experience:** Investigate low-rated categories and compare ratings with delivery time before deciding what to improve.
- **Category strategy:** Review the leading revenue categories alongside margin and inventory data before changing stock or marketing budgets.
- **Regional performance:** Compare order counts and average order value across regions to understand revenue differences.
- **Discount effectiveness:** Measure performance with and without discounts; revenue alone does not show profit.
- **Trend interpretation:** Verify the unusually broad date range before making claims about growth or seasonality.

## 8. Limitations

- The dataset contains 5,000 records and 12 columns; its coverage and representativeness are not established by this notebook.
- The date range runs from **2022-01-01 to 2035-09-09** in the supplied file. Validate future dates and data provenance before reporting historical trends.
- Revenue is analyzed as supplied. The notebook does not assume a revenue formula or calculate profit because cost and margin fields are not provided.
- Relationships between ratings, discounts, delivery times, and revenue are descriptive and do not prove causation.

## 9. How to run this project

1. Download `Internship.ipynb` and `sales_dataset.csv` into the same folder.
2. Open the notebook in Google Colab or Jupyter Notebook.
3. Run the cells from top to bottom.
4. If running locally, install dependencies from `requirements.txt`.

## Author

Add your name and GitHub profile here before publishing.
